# Solving the Bikerman model
We seek to solve the Bikerman finite-size model for a symmetric 1:1 electrolyte:
$$
\boxed{
\begin{aligned}
    \text{Bikerman}&\begin{cases}
    \varepsilon_{0}\varepsilon_{r}\dfrac{\mathrm{d}^{2}\phi}{\mathrm{d}x^{2}}
    &=
    \dfrac{2 e_{0} N_{\mathrm{A}} c_{\mathrm{b}}
    \sinh\left(\dfrac{e_{0}\phi}{k_{\mathrm{B}}T}\right)}
    {1 + 4r_{0}^{3}N_{\mathrm{A}}c_{\mathrm{b}}
    \sinh^{2}\left(\dfrac{e_{0}\phi}{2 k_{\mathrm{B}}T}\right)},
    &0 < x < +\infty  \\
    \phi\left(x\right) &= \phi_{\mathrm{M}} - \phi_{\mathrm{pzc}},
    &x = 0 \\
    \phi\left(x\right) &= 0,
    &x = +\infty
    \end{cases}
\end{aligned}
}
$$

Compared with the Gouy--Chapman model, the Bikerman model keeps the diffuse layer but introduces a finite ion / lattice size parameter $r_{0}$.  This produces a steric denominator that prevents the ion concentrations from diverging near highly charged surfaces.

# Parametric setup  ###

### The following lines of code set up the parameters and fundamental constants of the Bikerman model


In [ ]:
from dataclasses import dataclass
import math

# Fundamental constants
e = 1.602176634e-19       # C
kB = 1.380649e-23         # J/K
eps0 = 8.8541878128e-12   # F/m
NA = 6.02214076e23        # 1/mol


@dataclass
class Bikerman_SI_Params:
    r0: float                  # Ion / lattice size parameter, m

    eps_S: float = 78.5
    c_bulk: float = 1000.0
    T: float = 298.15

    tol: float = 1e-12
    maxiter: int = 200


def compute_beta(p: Bikerman_SI_Params) -> float:
    return 1.0 / (kB * p.T)


def compute_n_bulk(p: Bikerman_SI_Params) -> float:
    """
    Number density of each ion species for a symmetric 1:1 electrolyte.

    If c_bulk is the salt concentration in mol/m^3, then
    n_bulk = N_A c_bulk.
    """
    return NA * p.c_bulk


def compute_steric_parameter(p: Bikerman_SI_Params) -> float:
    """
    Bikerman steric parameter for symmetric 1:1 electrolyte:

        nu = 2 r0^3 n_bulk

    The denominator becomes

        1 + nu [cosh(beta e psi) - 1]

    or equivalently

        1 + 2 nu sinh^2(beta e psi / 2).
    """
    n_bulk = compute_n_bulk(p)
    return 2.0 * p.r0**3 * n_bulk


def compute_lambda_D(p: Bikerman_SI_Params) -> float:
    """
    Debye length for a symmetric 1:1 electrolyte.

    This is still useful as a reference length scale.
    """
    return math.sqrt(
        p.eps_S * eps0 * kB * p.T /
        (2.0 * e**2 * NA * p.c_bulk)
    )


def validate_params_si(p: Bikerman_SI_Params) -> None:
    if p.T <= 0:
        raise ValueError("T must be > 0")
    if p.eps_S <= 0:
        raise ValueError("eps_S must be > 0")
    if p.c_bulk <= 0:
        raise ValueError("c_bulk must be > 0")
    if p.r0 <= 0:
        raise ValueError("r0 must be > 0")

# Helper functions

### For stability, we define the following voltage conversion functions

$$
\begin{align*}
V_{\mathrm{T}} &= \frac{k_{\mathrm{B}}T}{e_{0}},
&U &= \frac{\phi}{V_{\mathrm{T}}},
&\phi &= U V_{\mathrm{T}} .
\end{align*}
$$

Here $U$ is the dimensionless electrostatic potential.

In [ ]:
def thermal_voltage(p: Bikerman_SI_Params) -> float:
    return kB * p.T / e


def V_to_U(p: Bikerman_SI_Params, V: float) -> float:
    return V / thermal_voltage(p)


def U_to_V(p: Bikerman_SI_Params, U: float) -> float:
    return U * thermal_voltage(p)

# Surface Free Charge Density and Double Layer Capacitance

## Surface Free Charge Density

For the Bikerman model, the first integral of the modified Poisson--Boltzmann equation gives
$$
\begin{aligned}
\sigma_{\mathrm{free}}^{\left(\mathrm{B}\right)}\left(\phi_{\mathrm{M}}-\phi_{\mathrm{pzc}}\right)
&=
\operatorname{sgn}\left(\phi_{\mathrm{M}}-\phi_{\mathrm{pzc}}\right)
\sqrt{
\frac{2\varepsilon_{0}\varepsilon_{r}}{r_{0}^{3}\beta}
\ln\left[
1 + 4r_{0}^{3}N_{\mathrm{A}}c_{\mathrm{b}}
\sinh^{2}\left(
\frac{e_{0}\left(\phi_{\mathrm{M}}-\phi_{\mathrm{pzc}}\right)}{2k_{\mathrm{B}}T}
\right)
\right]
},
\end{aligned}
$$
where
$$
\beta = \frac{1}{k_{\mathrm{B}}T}.
$$

## Double Layer Capacitance

The differential capacitance is obtained from
$$
\begin{aligned}
C_{\mathrm{B}}
&= \frac{\partial \sigma_{\mathrm{free}}^{\left(\mathrm{B}\right)}}
{\partial \left(\phi_{\mathrm{M}}-\phi_{\mathrm{pzc}}\right)} .
\end{aligned}
$$

The code below evaluates both the surface free charge density and the differential capacitance in SI units.

In [ ]:
import math


def sigma_free_B(p: Bikerman_SI_Params, V: float) -> float:
    validate_params_si(p)

    if abs(V) < p.tol:
        return 0.0

    beta = compute_beta(p)
    n_bulk = compute_n_bulk(p)

    U = beta * e * V

    argument = 4.0 * p.r0**3 * n_bulk * math.sinh(U / 2.0)**2

    prefactor = 2.0 * eps0 * p.eps_S / (p.r0**3 * beta)

    sigma_abs = math.sqrt(prefactor * math.log1p(argument))

    return math.copysign(sigma_abs, V)


def C_B(p: Bikerman_SI_Params, V: float) -> float:
    validate_params_si(p)

    beta = compute_beta(p)
    n_bulk = compute_n_bulk(p)

    # PZC limit
    if abs(V) < p.tol:
        return math.sqrt(
            2.0 * eps0 * p.eps_S * beta * e**2 * n_bulk
        )

    U = beta * e * V

    denominator = (
        1.0
        + 4.0 * p.r0**3 * n_bulk * math.sinh(U / 2.0)**2
    )

    log_argument = (
        4.0 * p.r0**3 * n_bulk * math.sinh(U / 2.0)**2
    )

    return (
        e * n_bulk * abs(math.sinh(U))
        / denominator
        * math.sqrt(
            2.0 * eps0 * p.eps_S * p.r0**3 * beta
            / math.log1p(log_argument)
        )
    )

# Solution scheme

This solution is based on first rewriting the Bikerman equation as a first-order equation for the potential profile.  In dimensionless form,
$$
U = \frac{e_{0}\phi}{k_{\mathrm{B}}T},
\qquad
X = \frac{x}{\lambda_{\mathrm{D}}},
\qquad
\nu = 2r_{0}^{3}N_{\mathrm{A}}c_{\mathrm{b}},
$$
and the Bikerman denominator is
$$
D(U)=1+2\nu\sinh^{2}\left(\frac{U}{2}\right).
$$

The first integral can be written as
$$
\begin{aligned}
\left|\frac{\mathrm{d}U}{\mathrm{d}X}\right|
&= \sqrt{\frac{2}{\nu}\ln D(U)} .
\end{aligned}
$$

Since the analytical profile is not as compact as the Gouy--Chapman expression, the code solves the profile by inverse integration,
$$
\begin{aligned}
X(U) &= \int_{U}^{U_{\mathrm{M}}}
\frac{\mathrm{d}u}
{\sqrt{\dfrac{2}{\nu}\ln D(u)}} .
\end{aligned}
$$

The resulting tabulated profile is then interpolated onto the requested spatial grid.  For the far-field tail, where the potential is already small, the code attaches a Debye--Hückel exponential decay.

In [ ]:
import math
import numpy as np


def bikerman_D_from_U(p: Bikerman_SI_Params, U) :
    """
    Bikerman finite-size denominator for symmetric 1:1 electrolyte:

        D(U) = 1 + 2 nu sinh^2(U/2)

    where

        nu = 2 r0^3 n_bulk.
    """
    U_arr = np.asarray(U, dtype=float)
    nu = compute_steric_parameter(p)

    return 1.0 + 2.0 * nu * np.sinh(0.5 * U_arr) ** 2


def bikerman_dU_dX_abs(p: Bikerman_SI_Params, U):
    U_arr = np.asarray(U, dtype=float)
    nu = compute_steric_parameter(p)

    D = bikerman_D_from_U(p, U_arr)
    out = np.sqrt((2.0 / nu) * np.log(D))

    return out


def bikerman_g_U(p: Bikerman_SI_Params, U):
    U_arr = np.asarray(U, dtype=float)
    val = bikerman_dU_dX_abs(p, U_arr)

    if np.any(val == 0.0):
        raise ZeroDivisionError("g(U) is singular at U = 0.")

    return 1.0 / val


def bikerman_h_bound_U(p: Bikerman_SI_Params, U_s: float, eps_U: float) -> float:
    validate_params_si(p)

    U_s = abs(float(U_s))
    eps_U = float(eps_U)

    if U_s <= 0.0:
        return np.inf
    if eps_U <= 0.0:
        raise ValueError("eps_U must be > 0.")
    if eps_U >= 2.0 * U_s:
        raise ValueError("eps_U must be smaller than 2 |U_M|.")

    nu = compute_steric_parameter(p)

    denom_small = 1.0 + 2.0 * nu * math.sinh(0.25 * eps_U) ** 2
    log_small = math.log(denom_small)

    log_large = math.log(1.0 + 2.0 * nu * math.sinh(0.5 * U_s) ** 2)

    term1 = (
        nu * math.sinh(U_s) / denom_small
    ) ** 2

    term2_inner = (
        2.0 * nu * math.cosh(U_s) / denom_small
        + term1
    )

    h_bound = (
        1.0
        / (4.0 * log_small ** (5.0 / 2.0))
        * (3.0 * term1 + 2.0 * log_large * term2_inner)
    )

    return h_bound


def estimate_delta_U_bikerman_profile(
    p: Bikerman_SI_Params,
    UM: float,
    eps_U: float = 1e-6,
) -> float:
    """
    Estimate the dimensionless potential spacing Delta U for solving the
    Bikerman Poisson equation by inverse integration.

    This is used for the potential profile U(X), not for capacitance.
    """
    validate_params_si(p)

    U_s = abs(float(UM))
    eps_U = float(eps_U)

    if U_s <= eps_U:
        return 0.0
    if eps_U <= 0.0:
        raise ValueError("eps_U must be > 0.")
    if eps_U >= 2.0 * U_s:
        raise ValueError("eps_U must be smaller than 2 |U_M|.")

    U_min = 0.5 * eps_U

    h_bound = bikerman_h_bound_U(p, U_s, eps_U)
    g_eps_half = bikerman_g_U(p, U_min)

    dU = math.sqrt(
        6.0
        * eps_U
        * g_eps_half
        / ((U_s - U_min) * h_bound)
    )

    return float(dU)


def solve_U_profile_from_UM(
    UM: float,
    p: Bikerman_SI_Params,
    eps_U: float = 1e-6,
    max_points: int = 1_000_000,
):
    validate_params_si(p)

    if eps_U <= 0.0:
        raise ValueError("eps_U must be > 0.")

    if abs(UM) <= eps_U:
        return (
            np.array([0.0]),
            np.array([0.0]),
            {
                "n_points": 1,
                "U_min": 0.5 * eps_U,
                "dU": 0.0,
                "X_max": 0.0,
                "message": "UM is already within the requested bulk tolerance.",
            },
        )

    sign = math.copysign(1.0, UM)

    U_s = abs(UM)
    U_min = 0.5 * eps_U

    dU_est = estimate_delta_U_bikerman_profile(
        p=p,
        UM=UM,
        eps_U=eps_U,
    )

    if dU_est <= 0.0 or not np.isfinite(dU_est):
        raise RuntimeError(f"Could not estimate a valid Delta U. Got dU = {dU_est}.")

    # The theoretical error-bound estimate can become extremely conservative
    # near U = 0. For plotting, enforce a maximum number of grid points.
    dU_min_allowed = (U_s - U_min) / max_points

    dU = max(dU_est, dU_min_allowed)

    n_steps = int(math.ceil((U_s - U_min) / dU))
    n_steps = max(n_steps, 1)

    used_grid_cap = dU > dU_est

    U_abs_grid = np.linspace(U_s, U_min, n_steps + 1)

    g_vals = bikerman_g_U(p, U_abs_grid)

    dU_grid = np.abs(np.diff(U_abs_grid))
    X_grid = np.zeros_like(U_abs_grid)
    X_grid[1:] = np.cumsum(
        0.5 * (g_vals[:-1] + g_vals[1:]) * dU_grid
    )

    U_grid = sign * U_abs_grid

    info = {
        "n_points": len(U_grid),
        "U_s": U_s,
        "U_min": U_min,
        "dU": dU,
        "dU_estimated": dU_est,
        "used_grid_cap": used_grid_cap,
        "X_max": X_grid[-1],
        "nu": compute_steric_parameter(p),
    }

    return X_grid, U_grid, info


def psi_Bikerman(
    x,
    p: Bikerman_SI_Params,
    V_M: float,
    phi_pzc: float = 0.0,
    eps_phi: float = 1e-2,
):
    validate_params_si(p)

    x_arr = np.asarray(x, dtype=float)
    scalar_input = np.ndim(x) == 0

    phi_s = V_M - phi_pzc

    if abs(phi_s) <= eps_phi:
        out = np.zeros_like(x_arr, dtype=float)
        return out.item() if scalar_input else out

    UM = V_to_U(p, phi_s)
    eps_U = abs(V_to_U(p, eps_phi))

    X_grid, U_grid, info = solve_U_profile_from_UM(
        UM,
        p,
        eps_U=eps_U,
        max_points=1_000_000,
    )

    lam_D = compute_lambda_D(p)
    x_profile = X_grid * lam_D
    V_profile = U_to_V(p, U_grid)

    # Interpolate inside the resolved Bikerman region.
    # Beyond x_max, attach the Debye-Huckel far-field tail.
    x_cut = x_profile[-1]
    V_cut = V_profile[-1]

    psi_interp = np.interp(
        x_arr,
        x_profile,
        V_profile,
        left=V_profile[0],
        right=V_cut,
    )

    psi_tail = V_cut * np.exp(-(x_arr - x_cut) / lam_D)

    psi = np.where(
        x_arr <= x_cut,
        psi_interp,
        psi_tail,
    )

    return psi.item() if scalar_input else psi


def bikerman_denominator_from_psi(p: Bikerman_SI_Params, psi):
    """
    Bikerman denominator for symmetric 1:1 electrolyte:

        D = 1 + 4 r0^3 n_b sinh^2(U/2)

    where U = e psi / kBT.
    """
    psi_arr = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi_arr)
    n_bulk = compute_n_bulk(p)

    return (
        1.0
        + 4.0 * p.r0**3 * n_bulk * np.sinh(0.5 * U) ** 2
    )


def cation_concentration_Bikerman(x, p: Bikerman_SI_Params, psi):
    """
    Cation concentration for pure symmetric 1:1 Bikerman model.

    Units are the same as p.c_bulk, i.e. mol/m^3.
    """
    psi_arr = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi_arr)
    D = bikerman_denominator_from_psi(p, psi_arr)

    return p.c_bulk * np.exp(-U) / D


def anion_concentration_Bikerman(x, p: Bikerman_SI_Params, psi):
    """
    Anion concentration for pure symmetric 1:1 Bikerman model.

    Units are the same as p.c_bulk, i.e. mol/m^3.
    """
    psi_arr = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi_arr)
    D = bikerman_denominator_from_psi(p, psi_arr)

    return p.c_bulk * np.exp(+U) / D

# Interactive Bikerman sweep

The cells below mirror the interactive setup from the Gouy--Chapman and Gouy--Chapman--Stern notebooks, but use the finite-size Bikerman diffuse-layer model. The same controls sweep one physical input while the remaining parameters stay fixed at their base values.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C_PER_M2_TO_UC_PER_CM2 = 100.0
F_PER_M2_TO_UF_PER_CM2 = 100.0


def make_bikerman_params_and_VM_from_input(input_name, value, base_params, base_VM):
    r0 = base_params.r0
    eps_S = base_params.eps_S
    c_bulk = base_params.c_bulk
    T = base_params.T
    V_M = base_VM

    if input_name == "Concentration":
        if value <= 0:
            raise ValueError("Concentration sweep values must be > 0 mM")
        c_bulk = value

    elif input_name == "Temperature":
        if value <= 0:
            raise ValueError("Temperature sweep values must be > 0 K")
        T = value

    elif input_name == "Bulk dielectric constant":
        if value <= 0:
            raise ValueError("Bulk dielectric constant sweep values must be > 0")
        eps_S = value

    elif input_name == "Ion / lattice size":
        if value <= 0:
            raise ValueError("Ion / lattice size sweep values must be > 0 nm")
        r0 = value * 1e-9

    elif input_name == "Applied Potential":
        V_M = value

    else:
        raise ValueError(f"Unsupported input_name: {input_name}")

    p = Bikerman_SI_Params(
        r0=r0,
        eps_S=eps_S,
        c_bulk=c_bulk,
        T=T,
        tol=base_params.tol,
        maxiter=base_params.maxiter,
    )
    validate_params_si(p)
    return p, V_M


def _bikerman_label(input_name, val):
    if input_name == "Concentration":
        return f"{val:g} mM"
    if input_name == "Temperature":
        return f"{val:g} K"
    if input_name == "Applied Potential":
        return f"{val:g} V"
    if input_name == "Bulk dielectric constant":
        return rf"$\varepsilon_S = {val:g}$"
    if input_name == "Ion / lattice size":
        return rf"$r_0 = {val:g}$ nm"
    return f"{input_name} = {val:g}"


def plot_bikerman_profiles_sigma_cap(
    input_name,
    input_values,
    V_M,
    x_max_nm,
    npts,
    base_concentration,
    base_temperature,
    base_eps_S,
    base_r0_nm,
    phi_min,
    phi_max,
    sigma_npts=250,
):
    if phi_min >= phi_max:
        raise ValueError("phi_min must be smaller than phi_max")
    if x_max_nm <= 0:
        raise ValueError("x_max_nm must be > 0")
    if npts < 3:
        raise ValueError("npts must be at least 3")
    if base_r0_nm <= 0:
        raise ValueError("base r0 / ion size must be > 0 nm")

    base_params = Bikerman_SI_Params(
        r0=base_r0_nm * 1e-9,
        eps_S=base_eps_S,
        c_bulk=base_concentration,
        T=base_temperature,
    )
    validate_params_si(base_params)

    x_nm = np.linspace(0.0, x_max_nm, npts)
    x_m = x_nm * 1e-9
    phi_grid = np.linspace(phi_min, phi_max, sigma_npts)

    fig = plt.figure(figsize=(12, 7), constrained_layout=True)
    gs = fig.add_gridspec(2, 6)

    ax1 = fig.add_subplot(gs[0, 0:2])
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax3 = fig.add_subplot(gs[0, 4:6])
    ax4 = fig.add_subplot(gs[1, 0:3])
    ax5 = fig.add_subplot(gs[1, 3:6])

    sweeping_applied_potential = input_name == "Applied Potential"

    ncurves = len(input_values)
    tvals = np.array([0.0]) if ncurves == 1 else np.linspace(0.0, 1.0, ncurves, endpoint=False)

    def blend_with_white(rgb, strength=0.65):
        rgb = np.array(rgb[:3], dtype=float)
        return tuple((1 - strength) * np.ones(3) + strength * rgb)

    def mix(c1, c2, t):
        c1 = np.array(c1, dtype=float)
        c2 = np.array(c2, dtype=float)
        return tuple((1 - t) * c1 + t * c2)

    start_gray = np.array([0.15, 0.15, 0.15])
    end_gray = np.array([0.65, 0.65, 0.65])

    potential_colors = (
        [tuple(start_gray)]
        if ncurves == 1
        else [mix(start_gray, end_gray, i / (ncurves - 1)) for i in range(ncurves)]
    )
    cation_colors = [blend_with_white((1.0, 0.0, 0.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    anion_colors = [blend_with_white((0.0, 0.0, 1.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    sigma_colors = [blend_with_white((0.0, 0.55, 0.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    cap_colors = [blend_with_white((0.5, 0.0, 0.5, 1.0), 0.25 + 0.65 * t) for t in tvals]

    for i, val in enumerate(input_values):
        p, V_M_this = make_bikerman_params_and_VM_from_input(
            input_name=input_name,
            value=val,
            base_params=base_params,
            base_VM=V_M,
        )

        psi = psi_Bikerman(x_m, p, V_M_this)
        ncat = cation_concentration_Bikerman(x_m, p, psi)
        nani = anion_concentration_Bikerman(x_m, p, psi)

        label = _bikerman_label(input_name, val)

        ax1.plot(x_nm, psi, label=label, color=potential_colors[i], lw=2.5)
        ax2.plot(x_nm, ncat, label=label, color=cation_colors[i], lw=2)
        ax3.plot(x_nm, nani, label=label, color=anion_colors[i], lw=2)

        sigma_vals = C_PER_M2_TO_UC_PER_CM2 * np.array([sigma_free_B(p, V) for V in phi_grid])
        cap_vals = F_PER_M2_TO_UF_PER_CM2 * np.array([C_B(p, V) for V in phi_grid])

        ax4.plot(phi_grid, sigma_vals, label=label, color=sigma_colors[i], lw=2)
        ax5.plot(phi_grid, cap_vals, label=label, color=cap_colors[i], lw=2)

    ax1.set_title("Electric potential")
    ax1.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax1.set_ylabel("Electric potential (V)", fontsize=12)

    ax2.set_title("Cation density")
    ax2.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax2.set_ylabel("Cation density (mM = mol/m$^3$)", fontsize=12)

    ax3.set_title("Anion density")
    ax3.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax3.set_ylabel("Anion density (mM = mol/m$^3$)", fontsize=12)

    ax4.set_title("Surface free charge density")
    ax4.set_xlabel(r"$\phi_M - \phi_{pzc}$ (V)", fontsize=14)
    ax4.set_ylabel(r"Surface free charge density ($\mu$C/cm$^2$)")

    ax5.set_title("Bikerman differential capacitance")
    ax5.set_xlabel(r"$\phi_M - \phi_{pzc}$ (V)", fontsize=14)
    ax5.set_ylabel(r"Double-layer capacitance ($\mu$F/cm$^2$)")

    for ax in (ax1, ax2, ax3, ax4, ax5):
        ax.grid(True)

    ax1.legend(fontsize=8, loc="lower left")
    ax2.legend(fontsize=8, loc="lower left")
    ax3.legend(fontsize=8, loc="lower left")

    if not sweeping_applied_potential:
        ax4.legend(fontsize=9)
        ax5.legend(fontsize=9)

    return fig


# Define error box


In [ ]:
import ipywidgets as widgets


def show_error(message):
    display(widgets.HTML(
        f"""
        <div style="
            color:#8a1f11;
            background:#fff3f0;
            border:1px solid #f0b4a8;
            border-radius:6px;
            padding:10px;
            font-size:14px;
            max-width:700px;
        ">
            <b>Error:</b> {message}
        </div>
        """
    ))


# Make User Interface


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

LABEL_WIDTH = "210px"
BOX_WIDTH = "150px"


def make_row(label_html, widget, fontsize="16px"):
    widget.layout = widgets.Layout(width=BOX_WIDTH)
    label = widgets.HTML(
        value=f"""
        <span style="font-size:{fontsize}; line-height:1.2;">
            {label_html}
        </span>
        """,
        layout=widgets.Layout(width=LABEL_WIDTH),
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(align_items="center"))


input_dropdown = widgets.Dropdown(
    options=[
        "Concentration",
        "Temperature",
        "Bulk dielectric constant",
        "Ion / lattice size",
        "Applied Potential",
    ],
    value="Ion / lattice size",
    layout=widgets.Layout(width=BOX_WIDTH),
)
input_row = make_row("Input:", input_dropdown)

values_text = widgets.Text(value="2, 3, 4", placeholder="comma-separated values")
values_row = make_row("Sweep:", values_text)

VM_box = widgets.FloatText(value=0.1)
VM_row = make_row("ϕ<sub>M</sub> − ϕ<sub>pzc</sub> (V):", VM_box)

phi_min_box = widgets.FloatText(value=-0.3)
phi_min_row = make_row("ϕ<sub>min</sub> (V):", phi_min_box)

phi_max_box = widgets.FloatText(value=0.3)
phi_max_row = make_row("ϕ<sub>max</sub> (V):", phi_max_box)

xmax_box = widgets.FloatText(value=5.0)
xmax_row = make_row("x<sub>max</sub> (nm):", xmax_box)

base_c_box = widgets.FloatText(value=10.0)
base_c_row = make_row("base c (mM):", base_c_box)

base_T_box = widgets.FloatText(value=298.15)
base_T_row = make_row("base T (K):", base_T_box)

base_eps_S_box = widgets.FloatText(value=78.5)
base_eps_S_row = make_row("base ε<sub>S</sub>:", base_eps_S_box)

base_r0_box = widgets.FloatText(value=1.3)
base_r0_row = make_row("base r<sub>0</sub> / ion size (nm):", base_r0_box)

plot_button = widgets.Button(
    description="Plot",
    button_style="success",
    layout=widgets.Layout(width="120px"),
)
plot_row = widgets.HBox([widgets.HTML(layout=widgets.Layout(width=LABEL_WIDTH)), plot_button])

out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))
npts = 300

base_header = widgets.HTML(
    """
    <div style="
        font-size:14px;
        font-weight:600;
        margin-top:8px;
        margin-bottom:4px;
    ">
        Base values used for parameters not being swept:
    </div>
    """
)


def update_visible_base_inputs(*args):
    swept = input_dropdown.value

    base_c_row.layout.display = "" if swept != "Concentration" else "none"
    base_T_row.layout.display = "" if swept != "Temperature" else "none"
    base_eps_S_row.layout.display = "" if swept != "Bulk dielectric constant" else "none"
    base_r0_row.layout.display = "" if swept != "Ion / lattice size" else "none"

    VM_row.layout.display = "none" if swept == "Applied Potential" else ""


def on_plot_clicked(b):
    out.clear_output(wait=True)
    plt.close("all")

    try:
        input_values = np.array(
            [float(v.strip()) for v in values_text.value.split(",") if v.strip()]
        )

        if input_values.size == 0:
            raise ValueError("Please enter at least one sweep value.")
        if xmax_box.value <= 0:
            raise ValueError("x max must be > 0 nm.")
        if npts < 3:
            raise ValueError("Points must be at least 3.")
        if phi_min_box.value >= phi_max_box.value:
            raise ValueError("phi min must be smaller than phi max.")
        if base_r0_box.value <= 0:
            raise ValueError("base r0 / ion size must be > 0 nm.")

        fig = plot_bikerman_profiles_sigma_cap(
            input_name=input_dropdown.value,
            input_values=input_values,
            V_M=VM_box.value,
            x_max_nm=xmax_box.value,
            npts=npts,
            base_concentration=base_c_box.value,
            base_temperature=base_T_box.value,
            base_eps_S=base_eps_S_box.value,
            base_r0_nm=base_r0_box.value,
            phi_min=phi_min_box.value,
            phi_max=phi_max_box.value,
            sigma_npts=npts,
        )

        with out:
            display(fig)
        plt.close(fig)

    except ValueError as err:
        plt.close("all")
        out.clear_output(wait=True)
        with out:
            show_error(f"Invalid input: {err}")

    except Exception as err:
        plt.close("all")
        out.clear_output(wait=True)
        with out:
            show_error(f"Unexpected error: {err}")


try:
    input_dropdown.unobserve(update_visible_base_inputs, names="value")
except Exception:
    pass

input_dropdown.observe(update_visible_base_inputs, names="value")

plot_button.on_click(on_plot_clicked, remove=True)
plot_button.on_click(on_plot_clicked)

update_visible_base_inputs()

controls = widgets.VBox(
    [
        input_row,
        values_row,
        VM_row,
        phi_min_row,
        phi_max_row,
        xmax_row,
        base_header,
        base_c_row,
        base_T_row,
        base_eps_S_row,
        base_r0_row,
        plot_row,
    ],
    layout=widgets.Layout(width="390px", min_width="390px"),
)

output_panel = widgets.Box(
    [out],
    layout=widgets.Layout(flex="1 1 0%", width="auto"),
)

ui = widgets.HBox(
    [controls, output_panel],
    layout=widgets.Layout(width="100%", align_items="flex-start"),
)

display(ui)

# Compare with experimental data

Source:
### Valette, G. (1982). Double layer on silver single crystal electrodes in contact with electrolytes having anions which are slightly specifically adsorbed: Part II. The (100) face. Journal of Electroanalytical Chemistry and Interfacial Electrochemistry, 138(1), 37–54

### Fig. 3


# Read the data


In [ ]:
from pathlib import Path
import numpy as np

VALETTE_FILES = {
    5.0: "Valette1982_5mM.txt",
    10.0: "Valette1982_10mM.txt",
    20.0: "Valette1982_20mM.txt",
    40.0: "Valette1982_40mM.txt",
    100.0: "Valette1982_100mM.txt",
}


def load_valette_txt(filepath):
    E_vals = []
    C_vals = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue
            if line.startswith("%") or line.startswith("#"):
                continue

            parts = line.replace(",", " ").split()
            if len(parts) < 2:
                continue

            try:
                E_vals.append(float(parts[0]))
                C_vals.append(float(parts[1]))
            except ValueError:
                # Allow a small amount of header text.
                continue

    if not E_vals:
        raise ValueError(f"No numeric data found in {filepath}")

    return np.array(E_vals, dtype=float), np.array(C_vals, dtype=float)


def find_valette_data_dir():
    """
    Try the folder layout used by the Gouy--Chapman notebooks, while
    also allowing the data folder to sit next to this notebook.
    """
    candidates = [
        Path.cwd() / "Valette1982",
        Path.cwd().parent / "Valette1982",
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/Valette1982"),
        Path("/mnt/data"),
    ]

    for directory in candidates:
        if all((directory / filename).exists() for filename in VALETTE_FILES.values()):
            return directory

    for directory in candidates:
        if any((directory / filename).exists() for filename in VALETTE_FILES.values()):
            return directory

    return candidates[0]


DATA_DIR = find_valette_data_dir()


def load_all_valette_data(data_dir=DATA_DIR, quiet=False):
    data = {}
    missing = []

    for conc_mM, filename in VALETTE_FILES.items():
        filepath = Path(data_dir) / filename

        if filepath.exists():
            E, C = load_valette_txt(filepath)
            data[conc_mM] = {"E": E, "C": C, "path": filepath}
        else:
            missing.append(filepath)

    if missing and not quiet:
        print("Missing Valette files:")
        for filepath in missing:
            print(f"  {filepath}")

    return data, missing


VALETTE_DATA, VALETTE_MISSING = load_all_valette_data()


# Plot Experimental and Bikerman together


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def make_bikerman_params_for_valette(
    concentration_mM,
    temperature,
    eps_S,
    r0_nm,
):
    p = Bikerman_SI_Params(
        r0=r0_nm * 1e-9,
        eps_S=eps_S,
        c_bulk=concentration_mM,
        T=temperature,
    )
    validate_params_si(p)
    return p


def plot_bikerman_valette_comparison(
    concentrations_mM,
    temperature=298.15,
    eps_S=78.5,
    r0_nm=1.3,
    phi_pzc_sce=-0.92,
    E_min=-1.48,
    E_max=-0.32,
    npts=350,
    show_data=True,
    show_missing_note=True,
):

    if E_min >= E_max:
        raise ValueError("E min must be smaller than E max.")
    if r0_nm <= 0:
        raise ValueError("Ion / lattice size r0 must be > 0 nm.")
    if eps_S <= 0:
        raise ValueError("Bulk/solution dielectric constant ε_S must be > 0.")
    if temperature <= 0:
        raise ValueError("Temperature must be > 0 K.")
    if npts < 3:
        raise ValueError("npts must be at least 3.")

    concentrations_mM = [float(c) for c in concentrations_mM]
    if len(concentrations_mM) == 0:
        raise ValueError("Please enter at least one concentration.")

    E_grid = np.linspace(E_min, E_max, npts)
    V_model = E_grid - phi_pzc_sce

    fig = plt.figure(figsize=(12, 8.5), constrained_layout=True)
    gs = fig.add_gridspec(1, 1)
    ax = fig.add_subplot(gs[0, 0])

    cmap = plt.cm.magma_r
    ncurves = len(concentrations_mM)
    colors = [cmap(0.65)] if ncurves == 1 else [cmap(t) for t in np.linspace(0.15, 0.90, ncurves)]

    for color, conc in zip(colors, concentrations_mM):
        p = make_bikerman_params_for_valette(
            concentration_mM=conc,
            temperature=temperature,
            eps_S=eps_S,
            r0_nm=r0_nm,
        )

        cap_model = F_PER_M2_TO_UF_PER_CM2 * np.array(
            [C_B(p, V) for V in V_model]
        )

        ax.plot(
            E_grid,
            cap_model,
            lw=2.6,
            color=color,
            solid_capstyle="round",
            label=rf"Bikerman {conc:g} mM",
        )

        if show_data and conc in VALETTE_DATA:
            data = VALETTE_DATA[conc]
            darker = tuple(np.clip(np.array(color[:3]) * 0.72, 0, 1))

            scatter_kwargs = dict(
                s=36,
                marker="o",
                facecolor=darker,
                edgecolor="white",
                linewidth=0.7,
                alpha=0.95,
                zorder=3,
            )

            ax.scatter(
                data["E"],
                data["C"],
                label=rf"Valette 1982 {conc:g} mM",
                **scatter_kwargs,
            )

    ax.axvline(phi_pzc_sce, color="0.35", lw=1.2, ls="--")
    ax.set_title("Bikerman capacitance compared with Valette 1982", fontsize=18, pad=12)
    ax.set_xlabel(r"ϕ$_{\mathrm{M}}$ (V)", fontsize=14)
    ax.set_ylabel(r"Differential capacitance ($\mu$F/cm$^2$)", fontsize=14)

    ax.grid(True, alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=11)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            fontsize=8.5,
            frameon=True,
            fancybox=True,
            framealpha=0.92,
            edgecolor="0.85",
            ncols=2,
        )

    if show_missing_note and show_data and VALETTE_MISSING:
        available = ", ".join(f"{c:g} mM" for c in sorted(VALETTE_DATA))
        if available:
            note = f"Loaded Valette data: {available}."
        else:
            note = (
                "No Valette data files were found. "
                "Run the Valette data-loading cell and check the Google Drive folder."
            )

        fig.text(
            0.01,
            0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="0.35",
        )

    return fig

# Interactive Valette comparison UI

Use this panel to tune the Bikerman finite-size parameter and compare the resulting capacitance curves to the Valette data.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

VAL_LABEL_WIDTH = "190px"
VAL_BOX_WIDTH = "150px"


def make_valette_row(label_html, widget, fontsize="16px"):
    widget.layout = widgets.Layout(width=VAL_BOX_WIDTH)
    label = widgets.HTML(
        value=f"""
        <span style="font-size:{fontsize}; line-height:1.2;">
            {label_html}
        </span>
        """,
        layout=widgets.Layout(width=VAL_LABEL_WIDTH),
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(align_items="center"))


valette_T_box = widgets.FloatText(value=298.15)
valette_T_row = make_valette_row("T (K):", valette_T_box)

valette_eps_S_box = widgets.FloatText(value=78.5)
valette_eps_S_row = make_valette_row("ε<sub>S</sub>:", valette_eps_S_box)

valette_r0_box = widgets.FloatText(value=1.3)
valette_r0_row = make_valette_row("r<sub>0</sub> / ion size (nm):", valette_r0_box)

valette_phi_pzc_box = widgets.FloatText(value=-0.92)
valette_phi_pzc_row = make_valette_row("ϕ<sub>pzc</sub> (V):", valette_phi_pzc_box)

valette_E_min_box = widgets.FloatText(value=-1.48)
valette_E_min_row = make_valette_row("ϕ<sub>min</sub> (V):", valette_E_min_box)

valette_E_max_box = widgets.FloatText(value=-0.32)
valette_E_max_row = make_valette_row("ϕ<sub>max</sub> (V):", valette_E_max_box)

valette_plot_button = widgets.Button(
    description="Plot comparison",
    button_style="success",
    layout=widgets.Layout(width="150px"),
)

valette_plot_row = widgets.HBox(
    [widgets.HTML(layout=widgets.Layout(width=VAL_LABEL_WIDTH)), valette_plot_button]
)

valette_out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))


def on_valette_bikerman_plot_clicked(b):
    valette_out.clear_output(wait=True)
    plt.close("all")

    try:
        concentrations = [5.0, 10.0, 20.0, 40.0, 100.0]

        if valette_T_box.value <= 0:
            raise ValueError("T must be > 0 K.")
        if valette_eps_S_box.value <= 0:
            raise ValueError("ε_S must be > 0.")
        if valette_r0_box.value <= 0:
            raise ValueError("r0 / ion size must be > 0 nm.")
        if valette_E_min_box.value >= valette_E_max_box.value:
            raise ValueError("ϕ_min must be smaller than ϕ_max.")

        fig = plot_bikerman_valette_comparison(
            concentrations_mM=concentrations,
            temperature=valette_T_box.value,
            eps_S=valette_eps_S_box.value,
            r0_nm=valette_r0_box.value,
            phi_pzc_sce=valette_phi_pzc_box.value,
            E_min=valette_E_min_box.value,
            E_max=valette_E_max_box.value,
            npts=350,
        )

        with valette_out:
            display(fig)
        plt.close(fig)

    except ValueError as err:
        plt.close("all")
        valette_out.clear_output(wait=True)
        with valette_out:
            show_error(f"Invalid input: {err}")

    except Exception as err:
        plt.close("all")
        valette_out.clear_output(wait=True)
        with valette_out:
            show_error(f"Unexpected error: {err}")


try:
    valette_plot_button.on_click(on_valette_bikerman_plot_clicked, remove=True)
except Exception:
    pass

valette_plot_button.on_click(on_valette_bikerman_plot_clicked)

valette_controls = widgets.VBox(
    [
        valette_T_row,
        valette_eps_S_row,
        valette_r0_row,
        valette_phi_pzc_row,
        valette_E_min_row,
        valette_E_max_row,
        valette_plot_row,
    ],
    layout=widgets.Layout(width="390px", min_width="390px"),
)

valette_output_panel = widgets.Box(
    [valette_out],
    layout=widgets.Layout(flex="1 1 0%", width="auto"),
)

valette_ui = widgets.HBox(
    [valette_controls, valette_output_panel],
    layout=widgets.Layout(width="100%", align_items="flex-start"),
)

display(valette_ui)